In [ ]:
!pip install -q gradio gradio_client

In [ ]:
import gradio as gr
from gradio_client import Client, handle_file
import shutil
import os

def virtual_tryon(person_image, cloth_image):
    if person_image is None or cloth_image is None:
        return None, "⚠️ Please upload both images!"

    print("🚀 VirtualFit started...")

    try:
        client = Client("yisol/IDM-VTON")

        result = client.predict(
            dict={"background": handle_file(person_image), "layers": [], "composite": None},
            garm_img=handle_file(cloth_image),
            garment_des="clothing",
            is_checked=True,
            is_checked_crop=False,
            denoise_steps=30,
            seed=42,
            api_name="/tryon"
        )

        output_path = result[0]
        return output_path, "✅ Successfully completed!"

    except Exception as e:
        return None, f"❌ An error occurred: {str(e)}"

css = """
.container {max-width: 900px; margin: auto; padding-top: 20px;}
h1 {text-align: center; color: #4A90E2; font-family: 'Helvetica', sans-serif;}
#run_btn {background-color: #4A90E2; color: white; font-weight: bold;}
"""

with gr.Blocks(theme=gr.themes.Soft(), css=css) as demo:

    with gr.Row():
        gr.Markdown("# 👕 VirtualFit: AI Fitting Room")

    gr.Markdown("Upload a photo of a person and a clothing item. The AI will virtually try on the clothes.")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 1. Upload Images")
            input_person = gr.Image(label="🧍‍♂️ Person Image (Model)", type="filepath", height=300)
            input_cloth = gr.Image(label="👕 Clothing Image", type="filepath", height=300)

            run_button = gr.Button("✨ Try On (Run)", elem_id="run_btn")

        with gr.Column():
            gr.Markdown("### 2. Result")
            output_image = gr.Image(label="Generated Image", show_label=True)
            status_text = gr.Textbox(label="Status", interactive=False)

    run_button.click(
        fn=virtual_tryon,
        inputs=[input_person, input_cloth],
        outputs=[output_image, status_text]
    )

    gr.Markdown("---")
    gr.Markdown("Developed by **VirtualFit Team** (Shukrullo, Akmaljon, Sirojiddin) for Educational Purpose.")

demo.launch(share=True, debug=True)